In [2]:
import nflreadpy as nfl
import pandas as pd
import json

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

# ── PLAYER STATS ──
print("Loading player stats...")
stats = nfl.load_player_stats(YEARS)
df = stats.to_pandas()

# Regular season only
df = df[df['season_type'] == 'REG'].copy()
print(f"Total rows: {len(df)}")
print(f"Seasons: {sorted(df['season'].unique())}")

Loading player stats...
Total rows: 107356
Seasons: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


In [3]:
print("Loading schedules...")
sched = nfl.load_schedules(YEARS)
sched_df = sched.to_pandas()
sched_df = sched_df[sched_df['game_type'] == 'REG'][['season', 'week', 'home_team', 'away_team']].copy()

# Build (season, week, team) -> location lookup
location_lookup = {}
for _, row in sched_df.iterrows():
    location_lookup[(int(row['season']), int(row['week']), row['home_team'])] = 'home'
    location_lookup[(int(row['season']), int(row['week']), row['away_team'])] = 'away'

print(f"Schedule entries: {len(location_lookup)}")

Loading schedules...
Schedule entries: 3230


In [4]:
def export_position(df, position, carries_min=0, targets_min=0, attempts_min=0):
    pos_df = df[df['position'] == position].copy()
    
    # Filter to players who actually played
    if attempts_min:
        pos_df = pos_df[pos_df['attempts'] >= attempts_min]
    elif targets_min:
        pos_df = pos_df[(pos_df['targets'] >= targets_min) | (pos_df['carries'] >= carries_min)]
    elif carries_min:
        pos_df = pos_df[pos_df['carries'] >= carries_min]

    # Rename columns to match what our HTML expects
    pos_df = pos_df.rename(columns={
        'team': 'recent_team',
        'passing_interceptions': 'interceptions',
    })

    # Add home/away
    def get_location(row):
        return location_lookup.get((int(row['season']), int(row['week']), row['recent_team']), 'home')
    pos_df['location'] = pos_df.apply(get_location, axis=1)

    # Select and clean columns
    base = ['season', 'week', 'player_id', 'player_name', 'player_display_name', 
            'recent_team', 'opponent_team', 'location', 'fantasy_points', 'fantasy_points_ppr']
    
    pos_cols = {
        'QB': ['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions',
               'passing_epa', 'passing_air_yards', 'pacr', 'carries', 'rushing_yards', 'rushing_tds'],
        'RB': ['carries', 'rushing_yards', 'rushing_tds', 'targets', 'receptions',
               'receiving_yards', 'receiving_tds'],
        'WR': ['targets', 'receptions', 'receiving_yards', 'receiving_tds',
               'carries', 'rushing_yards'],
        'TE': ['targets', 'receptions', 'receiving_yards', 'receiving_tds'],
    }

    keep = [c for c in base + pos_cols[position] if c in pos_df.columns]
    pos_df = pos_df[keep].copy()

    # Round floats
    float_cols = pos_df.select_dtypes(include='float').columns
    pos_df[float_cols] = pos_df[float_cols].round(2)
    pos_df = pos_df.sort_values(['player_display_name', 'season', 'week'])
    pos_df = pos_df.where(pd.notnull(pos_df), None)

    records = pos_df.to_dict(orient='records')
    filename = f'{position.lower()}_data.json'
    with open(filename, 'w') as f:
        json.dump(records, f)
    
    print(f"{position}: {len(records)} records, {pos_df['player_display_name'].nunique()} players → {filename}")

export_position(df, 'QB', attempts_min=1)
export_position(df, 'RB', carries_min=1, targets_min=1)
export_position(df, 'WR', targets_min=1)
export_position(df, 'TE', targets_min=1)

QB: 3707 records, 143 players → qb_data.json
RB: 8221 records, 317 players → rb_data.json
WR: 14547 records, 497 players → wr_data.json
TE: 7312 records, 264 players → te_data.json


In [5]:
import json

for pos in ['qb', 'rb', 'wr', 'te']:
    with open(f'{pos}_data.json') as f:
        data = json.load(f)
    
    # Find players who have at least 1 game in 2025
    active_2025 = set(r['player_display_name'] for r in data if r['season'] == 2025)
    
    # Keep only those players across all seasons
    filtered = [r for r in data if r['player_display_name'] in active_2025]
    
    before_players = len(set(r['player_display_name'] for r in data))
    after_players = len(set(r['player_display_name'] for r in filtered))
    
    with open(f'{pos}_data.json', 'w') as f:
        json.dump(filtered, f)
    
    print(f"{pos.upper()}: {before_players} → {after_players} players, {len(filtered)} records")

QB: 143 → 76 players, 2915 records
RB: 317 → 141 players, 5164 records
WR: 497 → 240 players, 9793 records
TE: 264 → 138 players, 5404 records
